[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/44.%20The%20Carbon%20Footprint%20Modeling%20Problem/P44-Tier-3_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open In Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 44. The Carbon Footprint Modeling Problem

## Tier 3: Grasshopper Optimization Algorithm

### Goal

This notebook implements a population-based Grasshopper Optimization Algorithm (GOA) for a small carbon-footprint planning example. Each grasshopper represents a candidate network configuration: which facilities are active and which vehicle type is assigned to each route. Social interactions move the candidates through a bounded continuous search space; a decoding step turns each position into discrete decisions, and a weighted emissions score guides the search.

### Inputs and outputs

The inputs are facility data, vehicle data, route data, objective weights, and GOA parameters such as population size and iteration limit. The outputs are the best candidate found by the algorithm, its decoded decisions, a history of best-so-far fitness values, a textual analysis, and visualizations. The algorithm is stochastic, so results can vary between runs.

### Scope and modeling assumptions

- Each route is treated as one traversal per day. Its transportation emissions are its distance in kilometres multiplied by the assigned vehicle's emission factor in kilograms of CO₂-equivalent per kilometre.
- A facility contributes its stated daily emissions if it is active. Facility activation decisions and route vehicle assignments are independent in this model; no route-to-facility service constraint is imposed.
- The fitness is a weighted sum of transportation and facility emissions. It is a scalarized objective score, not a separate optimization of two objectives and not an unweighted physical emissions inventory.
- Facility and vehicle capacities, demand fulfilment, fixed costs, customer locations, and facility-to-customer distances are stored in the data but are not used in the objective or enforced as constraints. The decoded candidates are therefore bounded encoded decisions, not necessarily operationally feasible network plans.
- The model does not provide a baseline plan for calculating a carbon-reduction percentage, nor does it provide a verified target value for comparison.

### Solution sequence

The notebook defines the example data, explains the social-interaction and adaptive-coefficient functions, encodes and scores candidate solutions, initializes a population, and repeatedly updates its positions. It then reports and plots the best-so-far candidate. This sequence makes the connection between the mathematical model, the GOA update, and the decoded decisions explicit.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

## 1. Representing the planning problem

A data model keeps each type of information together. A facility has an identifier, coordinates, daily emissions, capacity, and fixed cost; a vehicle has an emission factor and capacity; and a route has a distance, demand, and customer information. The GOA solution itself is represented by a continuous position vector, while its decoded decisions are discrete facility and vehicle choices.

For this example, the position vector has one coordinate for each facility and one coordinate for each route. Facility coordinates encode activation decisions; route coordinates encode vehicle choices. The additional data fields are retained in the example, even though the scope above specifies which fields contribute to this model's score.

In [ ]:
@dataclass
class Facility:
    """A facility and its modeled daily emissions."""

    id: str
    x: float
    y: float
    daily_emissions: float
    capacity: float
    fixed_cost: float


@dataclass
class Vehicle:
    """A transportation mode and its emissions per kilometre."""

    id: str
    vehicle_type: str
    emission_factor: float
    capacity: float
    cost_per_km: float


@dataclass
class Route:
    """A delivery route with demand and distance data."""

    id: str
    from_facility: str
    to_customer: str
    customer_x: float
    customer_y: float
    demand: float
    distance: float
    assigned_vehicle: str


@dataclass
class Grasshopper:
    """A candidate solution and its decoded decisions."""

    position: np.ndarray
    fitness: float
    active_facilities: List[str]
    vehicle_assignments: Dict[str, str]


@dataclass
class GOAProblem:
    """Data and objective weights for the GOA example."""

    facilities: List[Facility]
    vehicles: List[Vehicle]
    routes: List[Route]
    transport_weight: float = 0.6
    facility_weight: float = 0.4

## 2. Creating the concrete example

The example contains five facilities, three vehicle types, and eight delivery routes. Distances are in kilometres, facility emissions are in kilograms of CO₂-equivalent per day, and vehicle emission factors are in kilograms of CO₂-equivalent per kilometre. The routes are treated as one traversal each per day when transportation emissions are calculated.

In [ ]:
def create_concrete_example() -> GOAProblem:
    """Create the example facility, vehicle, and route data."""
    facilities = [
        Facility("F1", 10, 20, 120, 1000, 500),
        Facility("F2", 30, 25, 85, 1200, 400),
        Facility("F3", 50, 15, 200, 800, 600),
        Facility("F4", 70, 30, 150, 900, 550),
        Facility("F5", 90, 20, 95, 1100, 450),
    ]

    vehicles = [
        Vehicle("Truck", "Truck", 0.8, 100, 1.2),
        Vehicle("Rail", "Rail", 0.3, 500, 0.8),
        Vehicle("Ship", "Ship", 0.15, 1000, 0.5),
    ]

    routes = [
        Route("R1", "F1", "C1", 15, 35, 25, 45.3, "Truck"),
        Route("R2", "F2", "C2", 25, 45, 30, 52.1, "Rail"),
        Route("R3", "F3", "C3", 35, 25, 20, 38.7, "Truck"),
        Route("R4", "F4", "C4", 45, 40, 35, 41.2, "Rail"),
        Route("R5", "F5", "C5", 55, 30, 25, 33.8, "Ship"),
        Route("R6", "F1", "C6", 20, 50, 40, 58.9, "Truck"),
        Route("R7", "F2", "C7", 40, 35, 15, 28.4, "Rail"),
        Route("R8", "F3", "C8", 60, 45, 30, 47.6, "Ship"),
    ]

    return GOAProblem(facilities, vehicles, routes)


problem = create_concrete_example()
print(
    f"Problem created with {len(problem.facilities)} facilities, "
    f"{len(problem.vehicles)} vehicles, and {len(problem.routes)} routes."
)
print(
    f"Transport weight: {problem.transport_weight}; "
    f"facility weight: {problem.facility_weight}."
)

## 3. GOA functions and their interpretation

### Social interaction

Grasshoppers influence one another according to their separation. The social-interaction function controls how strongly a neighbour attracts or repels a candidate at a given distance:

$$
s(r) = f e^{-r/l} - e^{-r}.
$$

Here, $r$ is the Euclidean distance between two position vectors, while $f$ and $l$ control the interaction shape. The function can be positive or negative, allowing the social term to represent attraction or repulsion. In the implementation, `social_interaction_function` evaluates this expression.

### Adaptive coefficient

The coefficient decreases linearly over the run:

$$
c(t) = c_{\max} - \frac{t(c_{\max}-c_{\min})}{T},
\qquad 0 \leq t < T.
$$

Here, $t$ is the current iteration and $T$ is the iteration limit. A larger coefficient gives the position-update rule more influence; a smaller coefficient later in the run reduces the scale of its movement. The code implements this schedule in `adaptive_coefficient`.

The plots below show the shape of these two functions for the parameter values used by the example. They illustrate the update components; they do not represent optimization results.

In [ ]:
def social_interaction_function(
    r: float,
    f: float = 0.5,
    l: float = 1.5,
) -> float:
    """Evaluate the social interaction strength at distance r."""
    return f * np.exp(-r / l) - np.exp(-r)


def adaptive_coefficient(
    t: int,
    max_iter: int,
    c_max: float = 1.0,
    c_min: float = 0.00001,
) -> float:
    """Return the linearly decreasing GOA coefficient."""
    if max_iter <= 0:
        raise ValueError("max_iter must be positive.")
    return c_max - t * (c_max - c_min) / max_iter


distances = np.linspace(0.1, 10, 100)
interactions = [social_interaction_function(r) for r in distances]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(distances, interactions, color="tab:blue", linewidth=2)
axes[0].set_xlabel("Distance, r (position-space units)")
axes[0].set_ylabel("Social interaction strength, s(r)")
axes[0].set_title("Social Interaction Function")
axes[0].grid(True, alpha=0.3)

iterations = np.arange(0, 500)
coefficients = [adaptive_coefficient(t, 500) for t in iterations]
axes[1].plot(iterations, coefficients, color="tab:red", linewidth=2)
axes[1].set_xlabel("Iteration, t")
axes[1].set_ylabel("Adaptive coefficient, c(t)")
axes[1].set_title("Adaptive Coefficient Schedule")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 4. Decoding candidates and calculating fitness

The GOA moves continuous vectors, but the planning decisions are discrete. Decoding bridges those representations. The first five coordinates correspond to facilities: a value greater than 0.5 activates the facility. The remaining coordinates correspond to routes; each is mapped to one of the available vehicle types by dividing the unit interval into equal-width choices.

Let $A$ be the set of active facilities and let $v_r$ be the vehicle assigned to route $r$. The model scores a decoded candidate using the weighted objective

$$
J(A, v) = w_{\mathrm{transport}}
\sum_{r \in R} d_r e_{v_r}
+ w_{\mathrm{facility}}
\sum_{i \in A} E_i.
$$

Here, $R$ is the set of routes, $d_r$ is route $r$'s distance, $e_{v_r}$ is the assigned vehicle's emissions per kilometre, $E_i$ is facility $i$'s daily emissions, and the two $w$ terms are the objective weights. In the code, `calculate_fitness` evaluates this scalar score and the optimizer minimizes it. The weights combine the two components; consequently, the returned score should not be interpreted as an unweighted total emissions inventory.

The model does not require any facility to be active or link a route to an active facility. Those omissions are part of the stated scope and mean that a low-scoring candidate is not, by itself, evidence of an operationally feasible network.

In [ ]:
def decode_solution(
    position: np.ndarray,
    problem: GOAProblem,
) -> Tuple[List[str], Dict[str, str]]:
    """Decode a bounded position into facility and vehicle decisions."""
    expected_size = len(problem.facilities) + len(problem.routes)
    if len(position) != expected_size:
        raise ValueError(
            f"Expected a position of length {expected_size}, got {len(position)}."
        )
    if not problem.vehicles:
        raise ValueError("At least one vehicle type is required.")
    if np.any(position < 0) or np.any(position > 1):
        raise ValueError("Every position coordinate must lie in [0, 1].")

    facility_threshold = 0.5
    active_facilities = [
        facility.id
        for i, facility in enumerate(problem.facilities)
        if position[i] > facility_threshold
    ]

    vehicle_assignments = {}
    facility_offset = len(problem.facilities)
    for i, route in enumerate(problem.routes):
        scaled_choice = position[facility_offset + i] * len(problem.vehicles)
        vehicle_idx = min(int(scaled_choice), len(problem.vehicles) - 1)
        vehicle_assignments[route.id] = problem.vehicles[vehicle_idx].id

    return active_facilities, vehicle_assignments


def calculate_fitness(
    active_facilities: List[str],
    vehicle_assignments: Dict[str, str],
    problem: GOAProblem,
) -> float:
    """Calculate the weighted facility and transportation emissions score."""
    facility_dict = {facility.id: facility for facility in problem.facilities}
    vehicle_dict = {vehicle.id: vehicle for vehicle in problem.vehicles}
    route_dict = {route.id: route for route in problem.routes}

    unknown_facilities = set(active_facilities) - set(facility_dict)
    if unknown_facilities:
        raise ValueError(f"Unknown facility IDs: {sorted(unknown_facilities)}")
    if set(vehicle_assignments) != set(route_dict):
        raise ValueError("Every route must have exactly one vehicle assignment.")
    unknown_vehicles = set(vehicle_assignments.values()) - set(vehicle_dict)
    if unknown_vehicles:
        raise ValueError(f"Unknown vehicle IDs: {sorted(unknown_vehicles)}")

    facility_emissions = sum(
        facility_dict[facility_id].daily_emissions
        for facility_id in active_facilities
    )
    transport_emissions = sum(
        route_dict[route_id].distance
        * vehicle_dict[vehicle_id].emission_factor
        for route_id, vehicle_id in vehicle_assignments.items()
    )

    return (
        problem.transport_weight * transport_emissions
        + problem.facility_weight * facility_emissions
    )

## 5. Initializing the population and updating positions

A population is a set of candidate solutions. Initialization draws each coordinate uniformly from $[0,1]$, so every initial candidate lies within the search bounds. The position dimension is $D = |F| + |R|$, where $F$ is the set of facilities and $R$ is the set of routes.

For grasshopper $i$, coordinate $d$, and iteration $t$, the implementation updates the position according to

$$
x_{i,d}(t+1) = \operatorname{clip}_{[0,1]}\left(
c(t)\sum_{j \ne i}
\xi_{ijd}\,\frac{D}{2}\,s(r_{ij})
\frac{x_{j,d}(t)-x_{i,d}(t)}{r_{ij}}
+ c(t)x_{\mathrm{best},d}
\right),
$$

where $r_{ij}$ is the Euclidean distance between the complete position vectors of candidates $i$ and $j$, and $\xi_{ijd}$ is drawn uniformly from $[-1,1]$. The social term aggregates the influence of the other grasshoppers; the target term pulls toward the best candidate found so far. Clipping keeps every coordinate in the decoding interval. Pairs at zero distance contribute no social force, avoiding division by zero.

In the code, `initialize_population` creates and scores the initial candidates, while `update_grasshopper_position` applies the stated update, decodes the new vector, and recalculates its fitness. The coefficients `f` and `l` are passed to the social-interaction function so that the parameters printed by the optimizer govern the update.

In [ ]:
def initialize_population(
    pop_size: int,
    problem: GOAProblem,
) -> List[Grasshopper]:
    """Create random bounded candidates and evaluate their fitness."""
    if pop_size <= 0:
        raise ValueError("pop_size must be positive.")

    position_size = len(problem.facilities) + len(problem.routes)
    population = []

    for _ in range(pop_size):
        position = np.random.random(position_size)
        active_facilities, vehicle_assignments = decode_solution(position, problem)
        fitness = calculate_fitness(
            active_facilities,
            vehicle_assignments,
            problem,
        )
        population.append(
            Grasshopper(
                position=position,
                fitness=fitness,
                active_facilities=active_facilities,
                vehicle_assignments=vehicle_assignments,
            )
        )

    return population


def update_grasshopper_position(
    grasshopper: Grasshopper,
    population: List[Grasshopper],
    best_grasshopper: Grasshopper,
    c: float,
    problem: GOAProblem,
    grasshopper_idx: int,
    f: float = 0.5,
    l: float = 1.5,
) -> Grasshopper:
    """Apply the social and target terms to one candidate position."""
    new_position = np.zeros_like(grasshopper.position)
    dimension = len(grasshopper.position)

    # Distances use complete position vectors, so compute each once per neighbour.
    distances = [
        np.linalg.norm(grasshopper.position - other.position)
        for other in population
    ]

    for coordinate in range(dimension):
        social_force = 0.0

        for other_idx, other in enumerate(population):
            if other_idx == grasshopper_idx:
                continue

            distance = distances[other_idx]
            if distance > 0:
                interaction = social_interaction_function(distance, f, l)
                direction = (
                    other.position[coordinate]
                    - grasshopper.position[coordinate]
                ) / distance
                random_direction = np.random.random() * 2 - 1
                social_force += (
                    c
                    * random_direction
                    * (dimension / 2)
                    * interaction
                    * direction
                )

        target_force = c * best_grasshopper.position[coordinate]
        new_position[coordinate] = np.clip(
            social_force + target_force,
            0,
            1,
        )

    active_facilities, vehicle_assignments = decode_solution(new_position, problem)
    fitness = calculate_fitness(
        active_facilities,
        vehicle_assignments,
        problem,
    )

    return Grasshopper(
        position=new_position,
        fitness=fitness,
        active_facilities=active_facilities,
        vehicle_assignments=vehicle_assignments,
    )

## 6. Running the Grasshopper Optimization Algorithm

At each iteration, the optimizer calculates the adaptive coefficient, updates every member of the current population, and records the best fitness seen so far. It retains a separate best-so-far candidate, so the reported best value cannot worsen even when the newly updated population contains no improvement. The convergence history contains the initial best value followed by one best-so-far value per iteration.

For a population of size $N$, position dimension $D$, and $T$ iterations, the social updates require $O(TN^2D)$ work when pairwise distances are computed once per candidate pair. Fitness evaluation adds work proportional to the number of facilities and routes for each candidate. Memory use is $O(ND + T)$ for the current population and convergence history. These bounds describe the implemented loops; they are not an optimality guarantee.

In [ ]:
def grasshopper_optimization_algorithm(
    problem: GOAProblem,
    pop_size: int = 15,
    max_iter: int = 500,
    c_max: float = 1.0,
    c_min: float = 0.00001,
    f: float = 0.5,
    l: float = 1.5,
) -> Tuple[Grasshopper, List[float]]:
    """Run GOA and return the best candidate and best-so-far history."""
    if max_iter <= 0:
        raise ValueError("max_iter must be positive.")

    print("=" * 80)
    print("GRASSHOPPER OPTIMIZATION ALGORITHM FOR CARBON FOOTPRINT")
    print("=" * 80)
    print(f"Population size: {pop_size}")
    print(f"Maximum iterations: {max_iter}")
    print(f"Adaptive coefficient range: [{c_max}, {c_min}]")
    print(f"Social parameters: f={f}, l={l}")

    population = initialize_population(pop_size, problem)
    best_grasshopper = min(population, key=lambda candidate: candidate.fitness)
    convergence_history = [best_grasshopper.fitness]

    print(f"Initial best fitness score: {best_grasshopper.fitness:.3f}")
    print(f"Initial active facilities: {best_grasshopper.active_facilities}")

    for t in range(max_iter):
        c = adaptive_coefficient(t, max_iter, c_max, c_min)
        new_population = []

        for idx, grasshopper in enumerate(population):
            updated_grasshopper = update_grasshopper_position(
                grasshopper,
                population,
                best_grasshopper,
                c,
                problem,
                idx,
                f,
                l,
            )
            new_population.append(updated_grasshopper)

        population = new_population
        current_best = min(population, key=lambda candidate: candidate.fitness)
        if current_best.fitness < best_grasshopper.fitness:
            best_grasshopper = current_best

        convergence_history.append(best_grasshopper.fitness)

        if (t + 1) % 100 == 0:
            print(
                f"Iteration {t + 1:4d}: "
                f"best fitness score = {best_grasshopper.fitness:.3f}"
            )

    return best_grasshopper, convergence_history


best_solution, convergence = grasshopper_optimization_algorithm(
    problem,
    pop_size=15,
    max_iter=500,
    c_max=1.0,
    c_min=0.00001,
    f=0.5,
    l=1.5,
)

## 7. Analyzing the best candidate

The following analysis separates the facility and transportation components before verifying their weighted combination against the candidate's stored fitness. This check connects the reported decisions back to the objective equation. The convergence comparison describes improvement in the weighted score during this run; it does not imply a percentage reduction against an external baseline.

In [ ]:
def analyze_goa_solution(
    best_solution: Grasshopper,
    convergence: List[float],
    problem: GOAProblem,
) -> None:
    """Report the decoded candidate and verify its weighted fitness."""
    if not convergence:
        raise ValueError("The convergence history must not be empty.")

    facility_dict = {facility.id: facility for facility in problem.facilities}
    vehicle_dict = {vehicle.id: vehicle for vehicle in problem.vehicles}
    route_dict = {route.id: route for route in problem.routes}

    facility_emissions = sum(
        facility_dict[facility_id].daily_emissions
        for facility_id in best_solution.active_facilities
    )
    transport_emissions = 0.0
    vehicle_usage = {
        vehicle.id: {"count": 0, "emissions": 0.0, "distance": 0.0}
        for vehicle in problem.vehicles
    }

    for route_id, vehicle_id in best_solution.vehicle_assignments.items():
        route = route_dict[route_id]
        vehicle = vehicle_dict[vehicle_id]
        route_emissions = route.distance * vehicle.emission_factor
        transport_emissions += route_emissions
        vehicle_usage[vehicle_id]["count"] += 1
        vehicle_usage[vehicle_id]["emissions"] += route_emissions
        vehicle_usage[vehicle_id]["distance"] += route.distance

    weighted_total = (
        problem.transport_weight * transport_emissions
        + problem.facility_weight * facility_emissions
    )
    if not math.isclose(
        weighted_total,
        best_solution.fitness,
        rel_tol=1e-9,
        abs_tol=1e-9,
    ):
        raise AssertionError("The reported components do not match candidate fitness.")

    print("\n" + "=" * 80)
    print("GRASSHOPPER OPTIMIZATION RESULTS")
    print("=" * 80)
    print(f"Best weighted fitness score: {best_solution.fitness:.3f}")
    print(f"Active facilities: {best_solution.active_facilities}")
    print(f"Facility emissions component: {facility_emissions:.3f} kg CO₂e/day")
    print(f"Transportation emissions component: {transport_emissions:.3f} kg CO₂e/day")

    print("\nVehicle assignments:")
    for vehicle_id, usage in vehicle_usage.items():
        percentage = (
            100 * usage["count"] / len(problem.routes)
            if problem.routes
            else 0.0
        )
        print(
            f"{vehicle_dict[vehicle_id].vehicle_type}: "
            f"{usage['count']} routes ({percentage:.1f}%), "
            f"{usage['distance']:.1f} km, "
            f"{usage['emissions']:.3f} kg CO₂e/day"
        )

    print("\nWeighted objective components:")
    print(
        f"Transport: {problem.transport_weight} × "
        f"{transport_emissions:.3f} kg CO₂e/day"
    )
    print(
        f"Facilities: {problem.facility_weight} × "
        f"{facility_emissions:.3f} kg CO₂e/day"
    )
    print(f"Verified weighted fitness score: {weighted_total:.3f}")

    initial_fitness = convergence[0]
    final_fitness = convergence[-1]
    improvement = initial_fitness - final_fitness
    improvement_percentage = (
        100 * improvement / abs(initial_fitness)
        if not math.isclose(initial_fitness, 0.0, abs_tol=1e-12)
        else float("nan")
    )

    print("\nBest-so-far score over the run:")
    print(f"Initial best score: {initial_fitness:.3f}")
    print(f"Final best score: {final_fitness:.3f}")
    print(f"Absolute score improvement: {improvement:.3f}")
    if math.isfinite(improvement_percentage):
        print(f"Relative score improvement: {improvement_percentage:.2f}%")
    print(
        "This comparison is relative to the run's initial best score, "
        "not to an external baseline plan."
    )


analyze_goa_solution(best_solution, convergence, problem)

## 8. Visualizing the search and candidate

The convergence plot shows the best-so-far weighted score, not the fitness of every grasshopper. The facility chart distinguishes active and inactive facilities while displaying their stated emissions. The vehicle chart shows the share of routes assigned to each mode, and the final chart compares the two emissions components with the weighted objective score. Because the score is weighted, its value is not the sum of the two component bars unless the weights and component values make it so.

In [ ]:
def visualize_goa_solution(
    best_solution: Grasshopper,
    convergence: List[float],
    problem: GOAProblem,
) -> None:
    """Plot convergence, facility decisions, mode assignments, and score components."""
    if not convergence:
        raise ValueError("The convergence history must not be empty.")

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle(
        "Grasshopper Optimization: Carbon Footprint Planning Example",
        fontsize=15,
        fontweight="bold",
    )

    ax1 = axes[0, 0]
    iteration_values = np.arange(len(convergence))
    ax1.plot(iteration_values, convergence, color="tab:blue", linewidth=2)
    ax1.scatter(
        [iteration_values[0], iteration_values[-1]],
        [convergence[0], convergence[-1]],
        color="tab:red",
        zorder=3,
        label="Initial and final best-so-far scores",
    )
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Weighted fitness score")
    ax1.set_title("Best-So-Far Convergence")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2 = axes[0, 1]
    facility_labels = [facility.id for facility in problem.facilities]
    facility_values = [
        facility.daily_emissions for facility in problem.facilities
    ]
    facility_colors = [
        "tab:green" if facility.id in best_solution.active_facilities else "tab:gray"
        for facility in problem.facilities
    ]
    ax2.bar(facility_labels, facility_values, color=facility_colors, alpha=0.85)
    ax2.set_xlabel("Facility")
    ax2.set_ylabel("Daily emissions when active (kg CO₂e/day)")
    ax2.set_title("Facility Decisions")
    ax2.grid(True, alpha=0.3, axis="y")
    ax2.bar([], [], color="tab:green", label="Active")
    ax2.bar([], [], color="tab:gray", label="Inactive")
    ax2.legend()

    ax3 = axes[1, 0]
    vehicle_dict = {vehicle.id: vehicle for vehicle in problem.vehicles}
    vehicle_counts = {vehicle.id: 0 for vehicle in problem.vehicles}
    for vehicle_id in best_solution.vehicle_assignments.values():
        vehicle_counts[vehicle_id] += 1

    nonzero_usage = [
        (vehicle_id, count)
        for vehicle_id, count in vehicle_counts.items()
        if count > 0
    ]
    if nonzero_usage:
        labels = [vehicle_dict[vehicle_id].vehicle_type for vehicle_id, _ in nonzero_usage]
        sizes = [count for _, count in nonzero_usage]
        ax3.pie(
            sizes,
            labels=labels,
            autopct="%1.1f%%",
            startangle=90,
            textprops={"fontsize": 10},
        )
    else:
        ax3.text(0.5, 0.5, "No routes", ha="center", va="center")
    ax3.set_title("Route Assignments by Vehicle Type")

    ax4 = axes[1, 1]
    facility_dict = {facility.id: facility for facility in problem.facilities}
    route_dict = {route.id: route for route in problem.routes}
    facility_emissions = sum(
        facility_dict[facility_id].daily_emissions
        for facility_id in best_solution.active_facilities
    )
    transport_emissions = sum(
        route_dict[route_id].distance
        * vehicle_dict[vehicle_id].emission_factor
        for route_id, vehicle_id in best_solution.vehicle_assignments.items()
    )
    categories = ["Facility component", "Transportation component", "Weighted score"]
    values = [facility_emissions, transport_emissions, best_solution.fitness]
    bars = ax4.bar(
        categories,
        values,
        color=["tab:green", "tab:blue", "tab:orange"],
        alpha=0.85,
    )
    ax4.set_ylabel("Component emissions (kg CO₂e/day) or weighted score")
    ax4.set_title("Emissions Components and Objective Score")
    ax4.tick_params(axis="x", labelrotation=15)
    ax4.grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, values):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    fig.tight_layout()
    plt.show()


visualize_goa_solution(best_solution, convergence, problem)

## 9. Computational and methodological interpretation

The population-based search uses social interactions and a decreasing coefficient to explore candidate positions, then retains the best score encountered. It is a stochastic metaheuristic: it does not certify that the best candidate is globally optimal, and repeated runs may return different candidates. The convergence curve is evidence about one run only.

For the configured population size $N$, iteration limit $T$, and position dimension $D$, pairwise distances are computed once for each grasshopper-neighbour pair within an update. The social update therefore has leading work $O(TN^2D)$; candidate fitness calculations additionally scale with the number of facilities and routes. The population and position vectors require $O(ND)$ storage, and the history requires $O(T)$ storage. The initial population and each updated population are evaluated, for $N(T+1)$ candidate evaluations in total.

The method's result must be interpreted within the model's scope. In particular, facility activation is not linked to route service, and capacity and demand are not enforced. The weighted score is not an independently verified total footprint, and no external baseline is available for a carbon-reduction claim. Adding such operational constraints or testing larger instances would extend the model; those extensions are not implemented here.

In [ ]:
def algorithm_performance_analysis(
    convergence: List[float],
    problem: GOAProblem,
    pop_size: int,
    max_iter: int,
) -> None:
    """Summarize run dimensions, evaluation count, and observed score change."""
    if not convergence:
        raise ValueError("The convergence history must not be empty.")

    n_facilities = len(problem.facilities)
    n_routes = len(problem.routes)
    dimension = n_facilities + n_routes
    initial_fitness = convergence[0]
    final_fitness = convergence[-1]
    absolute_improvement = initial_fitness - final_fitness

    print("=" * 80)
    print("GRASSHOPPER OPTIMIZATION RUN SUMMARY")
    print("=" * 80)
    print("Search characteristics:")
    print("- Population-based search with social-interaction updates.")
    print("- A linearly decreasing coefficient scales movement over iterations.")
    print("- The objective is a weighted scalar score of emissions components.")
    print("- The stochastic search does not provide an optimality certificate.")

    print("\nRun dimensions:")
    print(f"Population size: {pop_size}")
    print(f"Maximum iterations: {max_iter}")
    print(f"Position dimension: {dimension} ({n_facilities} facilities + {n_routes} routes)")
    print(f"Candidate evaluations: {pop_size * (max_iter + 1)}")
    print(f"Social-update leading work: O(T × N² × D), with N={pop_size}, D={dimension}")
    print("Population and position storage: O(N × D)")

    print("\nObserved best-so-far score change:")
    print(f"Initial best score: {initial_fitness:.3f}")
    print(f"Final best score: {final_fitness:.3f}")
    print(f"Absolute improvement in weighted score: {absolute_improvement:.3f}")
    print(
        "The score change summarizes this stochastic run; it is not a "
        "comparison with a verified optimum or baseline plan."
    )


algorithm_performance_analysis(
    convergence,
    problem,
    pop_size=15,
    max_iter=500,
)

## 10. Summary

This notebook represents a carbon-footprint planning candidate as a bounded continuous vector, decodes it into facility activation and route-to-vehicle decisions, and evaluates it with a weighted emissions objective. The Grasshopper Optimization Algorithm updates a population using social interactions, target attraction toward the best-so-far candidate, and a decreasing adaptive coefficient. The analysis and plots connect the resulting candidate and convergence history to the model's objective.

The result is a heuristic candidate for the stated simplified model, not a guaranteed optimum or a complete operational network plan. Interpretation should account for the omitted capacity, demand, service-linkage, cost, and baseline elements described in the scope.